Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_lstm_pollution.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd,e
date,,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,NaN
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,NaN
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,NaN
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,NaN
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 0])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 7)
Dimensiones de Y: (43788, 1)


In [9]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777
          nan]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536
          nan]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295
          nan]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181
          nan]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894
          nan]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698
          nan]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584
          nan]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347
          nan]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356
          nan]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241
          nan]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449
          nan]
 [ 0.57004095 -0.656257

Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 7)
Las dimensiones de testX son:  (8801, 12, 7)
Las dimensiones de valX son:  (4336, 12, 7)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

49/49 - 15s - 310ms/step - ia: 0.1821 - loss: 1.2528 - mae: 0.8704 - rmse: 1.1100 - smape: 1.6105 - val_ia: 0.2597 - val_loss: 0.5944 - val_mae: 0.6238 - val_rmse: 0.7398 - val_smape: 1.7749

Epoch 2/128                                           

49/49 - 1s - 15ms/step - ia: 0.1799 - loss: 1.2125 - mae: 0.8400 - rmse: 1.0933 - smape: 1.5955 - val_ia: 0.2574 - val_loss: 0.5689 - val_mae: 0.5987 - val_rmse: 0.7181 - val_smape: 1.8737

Epoch 3/128                                           

49/49 - 1s - 17ms/step - ia: 0.1854 - loss: 1.2101 - mae: 0.8279 - rmse: 1.0954 - smape: 1.5853 - val_ia: 0.2600 - val_loss: 0.5640 - val_mae: 0.5935 - val_rmse: 0.7136 - val_smape: 1.9144

Epoch 4/128                                           

49/49 - 1s - 23ms/step - ia: 0.1778 - loss: 1.2087 - mae: 0.8250 - rmse: 1.0975 - smape: 1.5946 - val_ia: 0.2620 - val_loss: 0.5607 - val_mae: 0.5899 - val_rmse: 0.7106 - val_smape: 1.9547

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

385/385 - 13s - 35ms/step - ia: 0.6728 - loss: 0.3816 - mae: 0.4254 - rmse: 0.5846 - smape: 0.8366 - val_ia: 0.3317 - val_loss: 0.1917 - val_mae: 0.3087 - val_rmse: 0.3771 - val_smape: 0.7941

Epoch 2/128                                                                       

385/385 - 8s - 21ms/step - ia: 0.7974 - loss: 0.1689 - mae: 0.2778 - rmse: 0.3879 - smape: 0.5981 - val_ia: 0.4180 - val_loss: 0.1106 - val_mae: 0.2394 - val_rmse: 0.2926 - val_smape: 0.6958

Epoch 3/128                                                                       

385/385 - 8s - 22ms/step - ia: 0.8641 - loss: 0.0847 - mae: 0.1907 - rmse: 0.2719 - smape: 0.4466 - val_ia: 0.4526 - val_loss: 0.1054 - val_mae: 0.2465 - val_rmse: 0.2930 - val_smape: 0.7307

Epoch 4/128                                                                       

385/385 - 9s - 24ms/step - ia: 0.8868 - loss: 0.0675 - mae: 0.1625 - rmse: 0.2373 - sma

...

385/385 - 11s - 28ms/step - ia: 0.2679 - loss: 1.2249 - mae: 0.8307 - rmse: 1.0676 - smape: 1.4883 - val_ia: 0.2463 - val_loss: 0.4982 - val_mae: 0.5466 - val_rmse: 0.6027 - val_smape: 1.6308

Epoch 2/128                                                                      

385/385 - 5s - 12ms/step - ia: 0.3567 - loss: 0.9558 - mae: 0.7249 - rmse: 0.9394 - smape: 1.3515 - val_ia: 0.3300 - val_loss: 0.2825 - val_mae: 0.3693 - val_rmse: 0.4250 - val_smape: 0.9218

Epoch 3/128                                                                      

385/385 - 5s - 12ms/step - ia: 0.6342 - loss: 0.4473 - mae: 0.4924 - rmse: 0.6439 - smape: 0.9053 - val_ia: 0.3917 - val_loss: 0.1521 - val_mae: 0.2740 - val_rmse: 0.3302 - val_smape: 0.7273

Epoch 4/128                                                                      

385/385 - 4s - 12ms/step - ia: 0.6962 - loss: 0.3294 - mae: 0.4240 - rmse: 0.5563 - smape: 0.8080 - val_ia: 0.3967 - val_loss: 0.1366 - val_mae: 0.2618 - val_rmse: 0.3168 - v

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



770/770 - 10s - 13ms/step - ia: 0.2686 - loss: 2.0942 - mae: 1.0156 - rmse: 1.2966 - smape: 1.6258 - val_ia: 0.1779 - val_loss: 0.6833 - val_mae: 0.6762 - val_rmse: 0.7093 - val_smape: 1.7277

Epoch 2/128                                                                       

770/770 - 4s - 5ms/step - ia: 0.2605 - loss: 1.9466 - mae: 1.0078 - rmse: 1.2754 - smape: 1.6307 - val_ia: 0.1786 - val_loss: 0.6781 - val_mae: 0.6730 - val_rmse: 0.7063 - val_smape: 1.7313

Epoch 3/128                                                                       

770/770 - 5s - 6ms/step - ia: 0.2730 - loss: 1.9229 - mae: 0.9933 - rmse: 1.2554 - smape: 1.6243 - val_ia: 0.1792 - val_loss: 0.6732 - val_mae: 0.6700 - val_rmse: 0.7034 - val_smape: 1.7351

Epoch 4/128                                                                       

770/770 - 4s - 5ms/step - ia: 0.2619 - loss: 2.0650 - mae: 1.0048 - rmse: 1.2826 - smape: 1.6308 - val_ia: 0.1798 - val_loss: 0.6684 - val_mae: 0.6671 - val_rmse: 0.7006 - v

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



49/49 - 4s - 78ms/step - ia: 0.2722 - loss: 1.0694 - mae: 0.7677 - rmse: 1.0290 - smape: 1.4434 - val_ia: 0.3201 - val_loss: 0.4313 - val_mae: 0.5131 - val_rmse: 0.6214 - val_smape: 1.5161

Epoch 2/128                                                                         

49/49 - 1s - 28ms/step - ia: 0.3757 - loss: 0.8167 - mae: 0.6747 - rmse: 0.8970 - smape: 1.3154 - val_ia: 0.4056 - val_loss: 0.3210 - val_mae: 0.4296 - val_rmse: 0.5310 - val_smape: 1.1959

Epoch 3/128                                                                         

49/49 - 1s - 25ms/step - ia: 0.5372 - loss: 0.5779 - mae: 0.5527 - rmse: 0.7520 - smape: 1.0570 - val_ia: 0.5518 - val_loss: 0.2181 - val_mae: 0.3301 - val_rmse: 0.4312 - val_smape: 0.8808

Epoch 4/128                                                                         

49/49 - 1s - 26ms/step - ia: 0.6753 - loss: 0.3748 - mae: 0.4478 - rmse: 0.6053 - smape: 0.8599 - val_ia: 0.6450 - val_loss: 0.1619 - val_mae: 0.2858 - val_rmse: 0.3775 - v

In [16]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}


In [17]:
#{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}